In [1]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv() # .env 파일을 환경변수로 등록

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

In [7]:
graph = Neo4jGraph(
    url = NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print("Langchain 과 Neo4j 연결 성공!")

Langchain 과 Neo4j 연결 성공!


In [8]:
graph.refresh_schema() # 스키마 새로고침 (그래프 구조 변경시 실행)

print(graph.schema) # 노드 레이블, 속성, 관계 유형과 방향

Node properties:
Student {age: INTEGER, name: STRING, student_id: INTEGER}
Course {name: STRING, course_id: INTEGER, level: STRING, duration: INTEGER}
Instructor {name: STRING, career: INTEGER, instructor_id: INTEGER}
Category {name: STRING, category_id: INTEGER}
Relationship properties:
ENROLLED_IN {score: INTEGER, enrolled_at: DATE}
The relationships:
(:Student)-[:ENROLLED_IN]->(:Course)
(:Course)-[:BELONGS_TO]->(:Category)
(:Instructor)-[:TEACHES]->(:Course)


In [9]:
query = """
MATCH (student:Student) - [ENROLLED_IN] -> (course:Course)
RETURN
    student.name AS student_name,
    course.name AS course_name,
    student.student_id AS student_id,
    course.course_id AS course_id    
ORDER BY student_id, course_id
"""

result = graph.query(query) # 쿼리를 받아 결과를 list(dict)로 반환

result

[{'student_name': '홍길동',
  'course_name': 'Python',
  'student_id': 1,
  'course_id': 101},
 {'student_name': '홍길동',
  'course_name': 'Data Analysis',
  'student_id': 1,
  'course_id': 104},
 {'student_name': '김영희',
  'course_name': 'Database',
  'student_id': 2,
  'course_id': 102},
 {'student_name': '김영희',
  'course_name': 'Machine Learning',
  'student_id': 2,
  'course_id': 103},
 {'student_name': '이민수',
  'course_name': 'Python',
  'student_id': 3,
  'course_id': 101},
 {'student_name': '이민수',
  'course_name': 'Data Analysis',
  'student_id': 3,
  'course_id': 104},
 {'student_name': '박서연',
  'course_name': 'Machine Learning',
  'student_id': 4,
  'course_id': 103},
 {'student_name': '박서연',
  'course_name': 'Deep Learning',
  'student_id': 4,
  'course_id': 105},
 {'student_name': '최준호',
  'course_name': 'Database',
  'student_id': 5,
  'course_id': 102},
 {'student_name': '최준호',
  'course_name': 'Langchain',
  'student_id': 5,
  'course_id': 106}]

In [10]:
llm = ChatOpenAI(
    model = os.getenv('OPENAI_MODEL'),
    temperature=0 # DB 의 정보만 가져올 것이므로 창의성은 0 (결정론적인 답변 = 일관적)
)

In [20]:
# LLM 과 Neo4j 를 연결하여 자연어 질문을 Cypher로 변환하고 답변하는 체인
chain = GraphCypherQAChain.from_llm(
    llm = llm,              # Cypher 생성 및 최종 답변 llm
    graph = graph,          # 참조할 Neo4j 그래프 객체
    verbose = True,          # 로그 출력
    validate_cypher= True,  # 생성된 Cypher 검증
    return_intermediate_steps = True, # 중간과정 함께 반환
    top_k = 10,                       # 조회결과 10개
    allow_dangerous_requests = True   # DB 쿼리 실행 위험성 확인
)

In [12]:
question = "Python 강의를 수강하는 학생을 알려줘."

response = chain.invoke({"query": question})

response

{'query': 'Python 강의를 수강하는 학생을 알려줘.',
 'result': 'Python 강의를 수강하는 학생은 홍길동과 이민수입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})\nRETURN s;\n"},
  {'context': [{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}},
    {'s': {'name': '이민수', 'student_id': 3, 'age': 24}}]}]}

In [ ]:
response['result'] # 최종 답변

'Python 강의를 수강하는 학생은 홍길동과 이민수입니다.'

In [14]:
response['intermediate_steps'] # 중간 과정 확인

[{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})\nRETURN s;\n"},
 {'context': [{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}},
   {'s': {'name': '이민수', 'student_id': 3, 'age': 24}}]}]

In [15]:
response.get('intermediate_steps') # 중간 과정 확인

[{'query': "MATCH (s:Student)-[:ENROLLED_IN]->(c:Course {name: 'Python'})\nRETURN s;\n"},
 {'context': [{'s': {'name': '홍길동', 'student_id': 1, 'age': 26}},
   {'s': {'name': '이민수', 'student_id': 3, 'age': 24}}]}]

In [16]:
# 질문 답변 chain 함수
def ask_graph(question : str) -> dict:
    if not question.strip():
        raise ValueError("질문을 입력하셔야 합니다!!")

    response = chain.invoke({"query": question})

    print(f"[질문] {question}")
    print(f"[최종 답변] {response['result']}")

    # 중간 과정 추가시

    for step in response.get('intermediate_steps', []):
        if 'query' in step:
            print(f"[생성된 Cypher] {step['query']}")
        if "context" in step:
            print(f"[조회 결과] {step['context']}")

    return response            

In [17]:
ask_graph("홍길동이 수강한 강의 담당 강사를 알려줘.")

[질문] 홍길동이 수강한 강의 담당 강사를 알려줘.
[최종 답변] 홍길동이 수강한 강의 담당 강사는 Capybara와 Alice입니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN DISTINCT i.name AS instructor_name;
[조회 결과] [{'instructor_name': 'Capybara'}, {'instructor_name': 'Alice'}]


{'query': '홍길동이 수강한 강의 담당 강사를 알려줘.',
 'result': '홍길동이 수강한 강의 담당 강사는 Capybara와 Alice입니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN DISTINCT i.name AS instructor_name;"},
  {'context': [{'instructor_name': 'Capybara'},
    {'instructor_name': 'Alice'}]}]}

In [21]:
ask_graph("홍길동이 수강한 강의 담당 강사를 알려줘.")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN DISTINCT i.name;
Full Context:
[{'i.name': 'Capybara'}, {'i.name': 'Alice'}]

> Finished chain.
[질문] 홍길동이 수강한 강의 담당 강사를 알려줘.
[최종 답변] 알 수 없습니다.
[생성된 Cypher] MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)
RETURN DISTINCT i.name;
[조회 결과] [{'i.name': 'Capybara'}, {'i.name': 'Alice'}]


{'query': '홍길동이 수강한 강의 담당 강사를 알려줘.',
 'result': '알 수 없습니다.',
 'intermediate_steps': [{'query': "MATCH (s:Student {name: '홍길동'})-[:ENROLLED_IN]->(c:Course)<-[:TEACHES]-(i:Instructor)\nRETURN DISTINCT i.name;"},
  {'context': [{'i.name': 'Capybara'}, {'i.name': 'Alice'}]}]}

In [19]:
ask_graph("수강생이 가장 많은 강의를 알려줘")

[질문] 수강생이 가장 많은 강의를 알려줘
[최종 답변] 수강생이 가장 많은 강의는 머신러닝으로, 수강생은 2명입니다.
[생성된 Cypher] MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)
RETURN c.name AS course_name, count(s) AS student_count
ORDER BY student_count DESC
LIMIT 1
[조회 결과] [{'course_name': 'Machine Learning', 'student_count': 2}]


{'query': '수강생이 가장 많은 강의를 알려줘',
 'result': '수강생이 가장 많은 강의는 머신러닝으로, 수강생은 2명입니다.',
 'intermediate_steps': [{'query': 'MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)\nRETURN c.name AS course_name, count(s) AS student_count\nORDER BY student_count DESC\nLIMIT 1'},
  {'context': [{'course_name': 'Machine Learning', 'student_count': 2}]}]}

In [25]:
ask_graph("홍길동과 같은 강의를 수강한 다른학생 알려줘")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (target:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other.student_id <> target.student_id
RETURN DISTINCT other.name AS student_name;
Full Context:
[{'student_name': '이민수'}]

> Finished chain.
[질문] 홍길동과 같은 강의를 수강한 다른학생 알려줘
[최종 답변] 이민수, but there is no information confirming that they took the same course as 홍길동.
[생성된 Cypher] MATCH (target:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)
WHERE other.student_id <> target.student_id
RETURN DISTINCT other.name AS student_name;
[조회 결과] [{'student_name': '이민수'}]


{'query': '홍길동과 같은 강의를 수강한 다른학생 알려줘',
 'result': '이민수, but there is no information confirming that they took the same course as 홍길동.',
 'intermediate_steps': [{'query': "MATCH (target:Student {name: '홍길동'})-[:ENROLLED_IN]->(course:Course)<-[:ENROLLED_IN]-(other:Student)\nWHERE other.student_id <> target.student_id\nRETURN DISTINCT other.name AS student_name;"},
  {'context': [{'student_name': '이민수'}]}]}